# Glyph minimal pairs: scoring the real evaluators (GPU)

Runs the whole glyph-pair study on a GPU machine (Colab, Kaggle or a lab server):

1. build the pairs;
2. score them with OCR, CLIPScore, multilingual CLIP, a VLM judge (plain and OCR-grounded), VQAScore and the learned probe;
3. run the pre-specified analysis.

**Hardware.** A T4 (16 GB) runs everything except the 7B judge. Use `Qwen/Qwen2.5-VL-3B-Instruct` there, and the 7B model on an L4, A100 or larger. With `VARIANTS = 1` (about 2,000 candidates) the judge takes roughly an hour on a T4. `VARIANTS = 3` is the planned final size.

**Private repo.** Set `GITHUB_TOKEN` in Colab secrets (or paste a read-only fine-grained token below).

In [ ]:
VARIANTS = 1                       # 3 for the final run
JUDGE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"   # "Qwen/Qwen2.5-VL-7B-Instruct" on >= 24 GB
RUN_VQASCORE = True                # needs ~16 GB for clip-flant5-xl
BRANCH = "claude/fervent-cannon-cmcsq4"
OUT = "outputs/glyph_pairs/run"

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = os.environ.get("GITHUB_TOKEN", "")
url = f"https://{token}@github.com/harsharajkumar-273/verifiedgen-india.git" if token else "https://github.com/harsharajkumar-273/verifiedgen-india.git"
if not os.path.exists("verifiedgen-india"):
    !git clone -q -b {BRANCH} {url} verifiedgen-india
%cd verifiedgen-india

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install -r requirements.txt open_clip_torch
if RUN_VQASCORE:
    !pip -q install t2v_metrics
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Build the pairs

In [ ]:
!python scripts/build_glyph_pairs.py fetch-fonts
!python scripts/build_glyph_pairs.py build --out-dir {OUT} --variants {VARIANTS} --contact-sheet 24
from IPython.display import Image, display
display(Image(f"{OUT}/contact_sheet.png", width=500))

## 2. Score
Each evaluator writes `scores/<name>.scores.jsonl`. Cells are independent; a failed one can be re-run alone.

In [ ]:
M = f"{OUT}/candidates.jsonl"; S = f"{OUT}/scores"
!python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators random image_stats
!python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators ocr_tesseract --option ocr_tesseract.cache_path={OUT}/ocr_cache.json
!python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators ocr_tesseract --option ocr_tesseract.crop=false --option ocr_tesseract.cache_path={OUT}/ocr_cache_full.json

In [ ]:
!python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators clipscore --clip-model openai/clip-vit-large-patch14
!python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators openclip

In [ ]:
!python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators vlm_judge --option vlm_judge.model_name={JUDGE_MODEL}
!python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators vlm_judge --option vlm_judge.model_name={JUDGE_MODEL} --option vlm_judge.ocr_grounded=true

In [ ]:
if RUN_VQASCORE:
    !python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators vqascore --option vqascore.model_name=clip-flant5-xl

### Learned probe (the trained fix)
Cross-fitted by concept. The `--holdout-script` runs test transfer to a script the probe never saw.

In [ ]:
!python scripts/glyph_probe.py extract --manifest {M} --out {OUT}/clip_features.npz
!python scripts/glyph_probe.py score --pairs {OUT}/pairs.jsonl --features {OUT}/clip_features.npz --output {S}/probe.scores.jsonl
for script in ["deva", "knda", "taml"]:
    !python scripts/glyph_probe.py score --pairs {OUT}/pairs.jsonl --features {OUT}/clip_features.npz --holdout-script {script} --output {S}/probe_holdout_{script}.scores.jsonl

### Optional: hosted judges
Any OpenAI-compatible endpoint that returns logprobs works, for example vLLM, OpenRouter, or OpenAI itself. Set the key in the environment first.
```
!OPENAI_API_KEY=... python scripts/score_images.py --manifest {M} --output-dir {S} --evaluators vlm_judge \
   --option vlm_judge.backend=openai --option vlm_judge.model_name=gpt-4o --option vlm_judge.base_url=https://api.openai.com/v1
```
TextPecker publishes its own scoring code. Run that, convert the output to `{"candidate_id","evaluator","score"}` JSONL, and drop it into `scores/`.

## 3. Analyse

In [ ]:
!python scripts/analyze_glyph_pairs.py --pairs {OUT}/pairs.jsonl --scores {S}/*.jsonl --output-dir {OUT}/results
# After native-reader validation, add:  --exclusions <exclusions.json>
from IPython.display import Markdown
display(Markdown(open(f"{OUT}/results/glyph_results.md").read()))

In [ ]:
!cd {OUT} && zip -qr ../glyph_run_results.zip scores results pairs.jsonl build_summary.json contact_sheet.png
print("Download", OUT.rsplit('/',1)[0] + "/glyph_run_results.zip")